In [1]:
import os
import re

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
path = "../data/train.csv"
df = pd.read_csv(path)
df = df.iloc[: , 1:]
df.head()

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
text = "comment_text"

n_labels = df[labels].sum(axis=1)
df["chars"] = df[text].str.len()
df["words"] = df[text].str.split().str.len()
df["is_toxic_any"] = n_labels > 0

reports = {
    "Overview": pd.Series({
        "rows": len(df),
        "missing_values": int(df.isnull().sum().sum()),
        "clean_comments": int((n_labels == 0).sum()),
        "toxic_any": int((n_labels > 0).sum()),
        "duplicate_texts": int(df[text].duplicated().sum()),
        "very_short_texts": int((df["chars"] < 5).sum()),
    }, name="value").to_frame(),

    "Label distribution": pd.DataFrame({
        "count": df[labels].sum(),
        "percent": (df[labels].mean() * 100).round(2),
    }),

    "Labels per comment (%)": (n_labels.value_counts(normalize=True).sort_index() * 100).round(2).rename("percent").to_frame(),

    "Length stats": df[["chars", "words"]].describe(percentiles=[.5, .9, .95, .99]).round(1),

    "Length by class": df.groupby("is_toxic_any")[["chars", "words"]].mean().round(1),

    "Label correlation": df[labels].corr().round(2),
}

for name, report in reports.items():
    print(f"\n=== {name} ===")
    display(report)


=== Overview ===


,value
rows,159571
missing_values,0
clean_comments,143346
toxic_any,16225
duplicate_texts,0
very_short_texts,0



=== Label distribution ===


,count,percent
toxic,15294,9.58
severe_toxic,1595,1.00
obscene,8449,5.29
threat,478,0.30
insult,7877,4.94
identity_hate,1405,0.88



=== Labels per comment (%) ===


,percent
0,89.83
1,3.99
2,2.18
3,2.64
4,1.10
5,0.24
6,0.02


... (2 outputs left out)


=== Length by class ===


,chars,words
is_toxic_any,,
False,404.3,68.9
True,303.3,52.7



=== Label correlation ===


,toxic,severe_toxic,obscene,threat,insult,identity_hate
toxic,1.00,0.31,0.68,0.16,0.65,0.27
severe_toxic,0.31,1.00,0.40,0.12,0.38,0.20
obscene,0.68,0.40,1.00,0.14,0.74,0.29
threat,0.16,0.12,0.14,1.00,0.15,0.12
insult,0.65,0.38,0.74,0.15,1.00,0.34
identity_hate,0.27,0.20,0.29,0.12,0.34,1.00


In [4]:
labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]


df["any_label"] = (df[labels].sum(axis=1) > 0).astype(int)

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["any_label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["any_label"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("تدريب:", len(train_df), "| تحقق:", len(val_df), "| اختبار:", len(test_df))


URL_RE      = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE  = re.compile(r"@\w+")
IP_RE       = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")
HTML_RE     = re.compile(r"<[^>]+>")
NEWLINE_RE  = re.compile(r"[\r\n\t]+")
APOS_RE     = re.compile(r"['’`]")
NONALPHA_RE = re.compile(r"[^a-z\s]")
SPACES_RE   = re.compile(r"\s+")

def clean_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = IP_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = NEWLINE_RE.sub(" ", text)
    text = APOS_RE.sub("", text)
    text = NONALPHA_RE.sub(" ", text)
    text = SPACES_RE.sub(" ", text).strip()
    return text


for part in (train_df, val_df, test_df):
    part["clean_text"] = part["comment_text"].apply(clean_text)

تدريب: 127656 | تحقق: 15957 | اختبار: 15958


In [5]:
train_df['comment_text'][0] ,train_df['clean_text'][0] 

('Message\nThanks for the message. Have fun out there!  22:01, 18 Feb 2005 (UTC)',
 'message thanks for the message have fun out there feb utc')

In [6]:
vec = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vec.fit_transform(train_df["clean_text"])
X_val = vec.transform(val_df["clean_text"])
X_test = vec.transform(test_df["clean_text"])

terms = np.array(vec.get_feature_names_out())
weights = np.asarray(X_train.sum(axis=0)).ravel()

shapes = pd.DataFrame(
    {"rows": [X_train.shape[0], X_val.shape[0], X_test.shape[0]],
     "features": [X_train.shape[1], X_val.shape[1], X_test.shape[1]]},
    index=["train", "validation", "test"],
)
density = X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100

top_terms = pd.DataFrame({
    "term": terms[np.argsort(weights)[::-1][:20]],
    "total_weight": np.sort(weights)[::-1][:20].round(2),
})

top_by_label = pd.DataFrame({
    col: terms[np.argsort(
        np.asarray(X_train[train_df[col].values == 1].mean(axis=0)).ravel()
    )[::-1][:10]]
    for col in labels
})

print("=== Matrix shapes ===")
display(shapes)
print(f"Non-zero density: {density:.3f}%")

print("\n=== Top 20 terms by total weight ===")
display(top_terms)

print("\n=== Top 10 terms per label ===")
display(top_by_label)

=== Matrix shapes ===


,rows,features
train,127656,10000
validation,15957,10000
test,15958,10000


Non-zero density: 0.395%

=== Top 20 terms by total weight ===


,term,total_weight
0,the,7090.87
1,to,5853.00
2,you,5704.60
3,and,4761.61
4,of,4683.12
5,is,4576.77
6,it,4123.44
7,that,4021.03
8,in,3762.86
9,for,3527.59



=== Top 10 terms per label ===


,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,you,fuck,fuck,you,you,gay
1,fuck,you,you,die,fuck,you
2,and,fucking,fucking,kill,fucking,fuck
3,the,bitch,shit,fucking,your,faggot
4,your,shit,your,your,are,fucking
5,to,suck,and,will,and,nigger
6,are,ass,the,fuck,the,are
7,fucking,your,bitch,and,bitch,and
8,is,asshole,my,to,shit,is
9,of,go,to,ass,to,shit


In [7]:
def evaluate(models, X, df):
    rows = []
    for col, clf in models.items():
        prob = clf.predict_proba(X)[:, 1]
        rows.append({
            "label": col,
            "AUC": round(roc_auc_score(df[col], prob), 4),
            "F1": round(f1_score(df[col], prob >= 0.5), 4),
        })
    res = pd.DataFrame(rows).set_index("label")
    res.loc["mean"] = res.mean().round(4)
    return res

models = {
    col: LogisticRegression(C=4, solver="liblinear", max_iter=1000).fit(X_train, train_df[col])
    for col in labels
}

print("=== Validation results ===")
display(evaluate(models, X_val, val_df))

os.makedirs("../outputs", exist_ok=True)
joblib.dump({"models": models, "vectorizer": vec}, "../outputs/tfidf_logreg.joblib")
print("Model saved to ../outputs/tfidf_logreg.joblib")

=== Validation results ===


,AUC,F1
label,,
toxic,0.9707,0.7600
severe_toxic,0.9849,0.3981
obscene,0.9826,0.7702
threat,0.9861,0.2388
insult,0.9783,0.6928
identity_hate,0.9717,0.4216
mean,0.9790,0.5469


Model saved to ../outputs/tfidf_logreg.joblib


In [8]:
saved = joblib.load("../outputs/tfidf_logreg.joblib")
models, vec = saved["models"], saved["vectorizer"]

print("=== Test results ===")
display(evaluate(models, X_test, test_df))

=== Test results ===


,AUC,F1
label,,
toxic,0.9733,0.7735
severe_toxic,0.9761,0.3740
obscene,0.9818,0.7891
threat,0.9945,0.4308
insult,0.9774,0.6808
identity_hate,0.9675,0.3713
mean,0.9784,0.5699


In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_curve

rows = []
for col, clf in models.items():
    p_val = clf.predict_proba(X_val)[:, 1]
    p, r, t = precision_recall_curve(val_df[col], p_val)
    f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-9)
    thr = float(t[np.argmax(f1)])

    p_test = clf.predict_proba(X_test)[:, 1]
    rows.append({
        "label": col,
        "threshold": round(thr, 3),
        "F1_default_0.5": round(f1_score(test_df[col], p_test >= 0.5), 4),
        "F1_tuned": round(f1_score(test_df[col], p_test >= thr), 4),
    })

threshold_results = pd.DataFrame(rows).set_index("label")
threshold_results.loc["mean"] = threshold_results.mean().round(4)
threshold_results

,threshold,F1_default_0.5,F1_tuned
label,,,
toxic,0.3640,0.7735,0.7826
severe_toxic,0.3140,0.3740,0.4830
obscene,0.2470,0.7891,0.8028
threat,0.0830,0.4308,0.5439
insult,0.2660,0.6808,0.7188
identity_hate,0.2170,0.3713,0.4250
mean,0.2485,0.5699,0.6260


In [9]:
data = [
    ("Thanks for the helpful edit, I really appreciate it.", "clean", ""),
    ("The article was last updated in 2019 and cites three sources.", "clean", ""),
    ("The movie has a scene where the hero kills the villain with a sword.", "clean", ""),
    ("You are an idiot and nobody wants you here.", "toxic", "toxic, insult"),
    ("What the hell is this crap, did you even read it, you moron?", "toxic", "toxic, obscene, insult"),
    ("I will find you and hurt you.", "toxic", "toxic, threat"),
    ("You worthless piece of garbage, go die in a hole, everyone hates you.", "toxic", "toxic, severe_toxic, insult"),
    ("People like you should not be allowed in this country, go back where you came from.", "toxic", "toxic, identity_hate"),
    ("Do the world a favor and stop editing, we all know you are not smart enough for this.", "toxic", "toxic, insult"),
    ("I disagree with your edit because the source you used is outdated.", "clean", ""),
]
examples = [d[0] for d in data]
expected_verdict = [d[1] for d in data]
expected_labels = [d[2] for d in data]


def predict(texts, threshold=0.5, clean_fn=None):
    cleaned = [clean_fn(t) for t in texts] if clean_fn else texts
    X = vec.transform(cleaned)

    probs = pd.DataFrame({col: clf.predict_proba(X)[:, 1] for col, clf in models.items()})
    max_prob = probs.max(axis=1)
    verdict = (max_prob >= threshold).map({True: "toxic", False: "clean"})
    triggered = probs.apply(lambda r: ", ".join(r.index[r >= threshold]), axis=1)

    out = pd.DataFrame({
        "text": [t[:55] for t in texts],
        "expected": expected_verdict,
        "predicted": verdict,
        "match": ["yes" if e == p else "NO" for e, p in zip(expected_verdict, verdict)],
        "expected_labels": expected_labels,
        "triggered": triggered,
        "max_prob": max_prob.round(3),
    })
    return out.join(probs.round(3))


results = predict(examples, threshold=0.5, clean_fn=clean_text)
display(results)
print(f"Verdict accuracy: {(results['match'] == 'yes').mean():.0%}")

,text,expected,predicted,match,expected_labels,triggered,max_prob,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,"Thanks for the helpful edit, I really apprecia...",clean,clean,yes,,,0.001,0.000,0.001,0.001,0.000,0.000,0.000
1,The article was last updated in 2019 and cites...,clean,clean,yes,,,0.011,0.011,0.000,0.004,0.000,0.009,0.001
2,The movie has a scene where the hero kills the...,clean,clean,yes,,,0.008,0.008,0.002,0.004,0.001,0.006,0.005
3,You are an idiot and nobody wants you here.,toxic,toxic,yes,"toxic, insult","toxic, insult",0.999,0.999,0.031,0.487,0.004,0.995,0.020
4,"What the hell is this crap, did you even read ...",toxic,toxic,yes,"toxic, obscene, insult","toxic, obscene, insult",1.000,1.000,0.033,0.980,0.002,0.964,0.016
5,I will find you and hurt you.,toxic,toxic,yes,"toxic, threat",toxic,0.524,0.524,0.013,0.040,0.086,0.116,0.006
6,"You worthless piece of garbage, go die in a ho...",toxic,toxic,yes,"toxic, severe_toxic, insult","toxic, obscene, insult",0.999,0.999,0.323,0.773,0.033,0.959,0.017
7,People like you should not be allowed in this ...,toxic,clean,NO,"toxic, identity_hate",,0.177,0.177,0.013,0.049,0.006,0.091,0.033
8,"Do the world a favor and stop editing, we all ...",toxic,clean,NO,"toxic, insult",,0.152,0.152,0.002,0.020,0.003,0.055,0.012
9,I disagree with your edit because the source y...,clean,clean,yes,,,0.018,0.018,0.001,0.008,0.001,0.007,0.003


Verdict accuracy: 80%


In [10]:
val_res = evaluate(models, X_val, val_df)
test_res = evaluate(models, X_test, test_df)

val_res.to_csv("../outputs/validation_results.csv")
test_res.to_csv("../outputs/test_results.csv")
results.to_csv("../outputs/examples_results.csv", index=False)